# Fleet Reliability Data Engineering Platform
# Objective:
Build vehicle-level reliability and service features for KPI analysis and anomaly detection.

This notebook creates vehicle-level reliability features from cleaned fleet datasets.

## Main tasks
- Build base vehicle features
- Aggregate service history
- Aggregate telemetry behavior
- Aggregate warranty claims
- Aggregate parts failure history
- Create recency, ratio, and risk features
- Save ML-ready and KPI-ready tables

In [1]:
# Importing Libraries
import pandas as pd
import numpy as np
import os

In [2]:
# Display Settings

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

In [7]:
# Define paths

processed_path = "data/processed"
feature_path = "data/featured"

os.makedirs(feature_path, exist_ok=True)

In [5]:
from google.colab import files
uploaded = files.upload()

Saving warranty_claims.csv to warranty_claims.csv
Saving vehicles.csv to vehicles.csv
Saving telemetry_summary.csv to telemetry_summary.csv
Saving service_records.csv to service_records.csv
Saving parts_failure.csv to parts_failure.csv


In [9]:
# Load Cleaned Datasets
vehicles = pd.read_csv("vehicles.csv")
service_records = pd.read_csv("service_records.csv")
telemetry_summary = pd.read_csv("telemetry_summary.csv")
warranty_claims = pd.read_csv("warranty_claims.csv")
parts_failure = pd.read_csv("parts_failure.csv")

In [10]:
# Shape Check
vehicles = pd.read_csv("vehicles.csv")
service_records = pd.read_csv("service_records.csv")
telemetry_summary = pd.read_csv("telemetry_summary.csv")
warranty_claims = pd.read_csv("warranty_claims.csv")
parts_failure = pd.read_csv("parts_failure.csv")

In [11]:
display(vehicles.head())

,vehicle_id,vin,model,model_year,battery_type,delivery_date,region,mileage,fleet_type
0,1,5YJ00000000000001,Model 3,2025,NCA,2020-04-11,Asia Pacific,1269,Retail
1,2,5YJ00000000000002,Model Y,2025,NCM,2019-01-22,North America,3505,Retail
2,3,5YJ00000000000003,Model 3,2023,LFP,2024-08-30,Europe,58323,Retail
3,4,5YJ00000000000004,Model 3,2019,LFP,2025-12-31,North America,87159,Retail
4,5,5YJ00000000000005,Model 3,2023,LFP,2023-04-29,Europe,44127,Internal Test


In [12]:
display(service_records.head())

,service_id,vehicle_id,service_date,service_center,issue_category,issue_subcategory,severity,repair_cost,downtime_days,repeat_issue_flag
0,1,34451,2024-12-17,Fremont,Battery,Range Drop,Medium,3058.68,2.6,0
1,2,14761,2025-04-02,New York,Brakes,ABS Fault,Medium,1119.33,4.9,1
2,3,29920,2023-03-14,Berlin,Suspension,Control Arm Fault,Low,2370.96,4.2,0
3,4,46989,2024-11-06,Austin,Software,Autopilot Error,High,745.19,3.6,1
4,5,6989,2025-06-16,Fremont,Electrical,Sensor Failure,Low,1151.60,4.5,1


In [13]:
display(telemetry_summary.head())

,telemetry_id,vehicle_id,date,battery_health,avg_motor_temp,avg_brake_temp,tire_pressure_alerts,software_version,warning_count
0,1,23741,2025-06-27,97.26,77.22,53.09,0,2025.8.1,7
1,2,17245,2025-03-18,98.23,72.30,62.43,2,2025.2.6,1
2,3,34950,2025-04-09,91.76,90.70,56.11,0,2025.2.6,1
3,4,49846,2025-05-31,95.87,81.47,61.18,0,2025.2.6,3
4,5,2253,2025-07-25,82.61,73.74,57.97,0,2025.12.4,2


In [14]:
display(warranty_claims.head())

,claim_id,vehicle_id,claim_date,component,claim_amount,claim_status
0,1,28788,2022-08-11,Wiring Harness,2325.68,Pending
1,2,46239,2023-11-26,BMS,7554.90,Approved
2,3,11206,2022-08-05,Drive Unit,5738.37,Pending
3,4,23091,2022-01-30,Drive Unit,7381.06,Approved
4,5,521,2023-11-23,Cabin Heater,2123.84,Approved


In [15]:
display(parts_failure.head())

,failure_id,vehicle_id,component,failure_date,failure_type,replaced_flag
0,1,5821,Cabin Heater,2025-08-25,Thermal Issue,1
1,2,13420,ABS Module,2025-06-16,Thermal Issue,1
2,3,44,Cooling Fan,2026-02-09,Software Error,1
3,4,27146,Motor Controller,2021-03-04,Software Error,0
4,5,48454,Wiring Harness,2021-02-05,Thermal Issue,0


In [16]:
# Create reference date
reference_date = pd.Timestamp("2026-03-01")
print("Reference date:", reference_date)

Reference date: 2026-03-01 00:00:00


In [17]:
# Create base vehicle feature table

vehicle_features = vehicles.copy()
vehicle_features.head()

,vehicle_id,vin,model,model_year,battery_type,delivery_date,region,mileage,fleet_type
0,1,5YJ00000000000001,Model 3,2025,NCA,2020-04-11,Asia Pacific,1269,Retail
1,2,5YJ00000000000002,Model Y,2025,NCM,2019-01-22,North America,3505,Retail
2,3,5YJ00000000000003,Model 3,2023,LFP,2024-08-30,Europe,58323,Retail
3,4,5YJ00000000000004,Model 3,2019,LFP,2025-12-31,North America,87159,Retail
4,5,5YJ00000000000005,Model 3,2023,LFP,2023-04-29,Europe,44127,Internal Test


In [19]:
# Vehicle Age Features
vehicle_features["delivery_date"] = pd.to_datetime(vehicle_features["delivery_date"])

vehicle_features["vehicle_age_days"] = (reference_date - vehicle_features["delivery_date"]).dt.days
vehicle_features["vehicle_age_months"] = (vehicle_features["vehicle_age_days"] / 30).round(1)
vehicle_features["vehicle_age_years"] = (vehicle_features["vehicle_age_days"] / 365).round(2)

vehicle_features[[
    "vehicle_id",
    "delivery_date",
    "vehicle_age_days",
    "vehicle_age_months",
    "vehicle_age_years"
]].head()

,vehicle_id,delivery_date,vehicle_age_days,vehicle_age_months,vehicle_age_years
0,1,2020-04-11,2150,71.7,5.89
1,2,2019-01-22,2595,86.5,7.11
2,3,2024-08-30,548,18.3,1.50
3,4,2025-12-31,60,2.0,0.16
4,5,2023-04-29,1037,34.6,2.84


In [20]:
# Mileage Band Feature

def mileage_band(mileage):
    if mileage < 15000:
        return "Low"
    elif mileage < 50000:
        return "Moderate"
    elif mileage < 100000:
        return "High"
    else:
        return "Very High"

vehicle_features["mileage_band"] = vehicle_features["mileage"].apply(mileage_band)
vehicle_features[["vehicle_id", "mileage", "mileage_band"]].head()

,vehicle_id,mileage,mileage_band
0,1,1269,Low
1,2,3505,Low
2,3,58323,High
3,4,87159,High
4,5,44127,Moderate


In [22]:
# Service recency helper flags

# Convert to datetime first
service_records["service_date"] = pd.to_datetime(service_records["service_date"])

# Calculate days since service
service_records["days_since_service"] = (reference_date - service_records["service_date"]).dt.days

# Create recency flags
service_records["service_last_30d"] = (service_records["days_since_service"] <= 30).astype(int)
service_records["service_last_90d"] = (service_records["days_since_service"] <= 90).astype(int)
service_records["service_last_180d"] = (service_records["days_since_service"] <= 180).astype(int)

# Preview
service_records[[
    "vehicle_id",
    "service_date",
    "days_since_service",
    "service_last_30d",
    "service_last_90d"
]].head()


,vehicle_id,service_date,days_since_service,service_last_30d,service_last_90d
0,34451,2024-12-17,439,0,0
1,14761,2025-04-02,333,0,0
2,29920,2023-03-14,1083,0,0
3,46989,2024-11-06,480,0,0
4,6989,2025-06-16,258,0,0


In [23]:
# Aggregate service features by vehicle

service_agg = service_records.groupby("vehicle_id").agg(
    total_services=("service_id", "count"),
    repairs_last_30d=("service_last_30d", "sum"),
    repairs_last_90d=("service_last_90d", "sum"),
    repairs_last_180d=("service_last_180d", "sum"),
    avg_repair_cost=("repair_cost", "mean"),
    max_repair_cost=("repair_cost", "max"),
    total_repair_cost=("repair_cost", "sum"),
    avg_downtime_days=("downtime_days", "mean"),
    max_downtime_days=("downtime_days", "max"),
    total_downtime_days=("downtime_days", "sum"),
    repeat_issue_count=("repeat_issue_flag", "sum"),
    last_service_date=("service_date", "max")
).reset_index()

service_agg.head()

,vehicle_id,total_services,repairs_last_30d,repairs_last_90d,repairs_last_180d,avg_repair_cost,max_repair_cost,total_repair_cost,avg_downtime_days,max_downtime_days,total_downtime_days,repeat_issue_count,last_service_date
0,1,9,0,1,1,1641.293333,5206.91,14771.64,2.633333,5.4,23.7,1,2025-12-14
1,2,7,0,1,2,1715.110000,3449.08,12005.77,2.457143,4.0,17.2,1,2026-01-01
2,3,4,0,0,2,1585.157500,2295.66,6340.63,4.000000,6.4,16.0,1,2025-11-20
3,4,3,0,0,0,1760.020000,3445.09,5280.06,4.366667,5.1,13.1,1,2024-12-25
4,5,2,0,0,0,721.215000,1353.66,1442.43,4.550000,4.8,9.1,0,2025-05-01


In [24]:
# Days since last service
service_agg["days_since_last_service"] = (reference_date - service_agg["last_service_date"]).dt.days
service_agg.head()


,vehicle_id,total_services,repairs_last_30d,repairs_last_90d,repairs_last_180d,avg_repair_cost,max_repair_cost,total_repair_cost,avg_downtime_days,max_downtime_days,total_downtime_days,repeat_issue_count,last_service_date,days_since_last_service
0,1,9,0,1,1,1641.293333,5206.91,14771.64,2.633333,5.4,23.7,1,2025-12-14,77
1,2,7,0,1,2,1715.110000,3449.08,12005.77,2.457143,4.0,17.2,1,2026-01-01,59
2,3,4,0,0,2,1585.157500,2295.66,6340.63,4.000000,6.4,16.0,1,2025-11-20,101
3,4,3,0,0,0,1760.020000,3445.09,5280.06,4.366667,5.1,13.1,1,2024-12-25,431
4,5,2,0,0,0,721.215000,1353.66,1442.43,4.550000,4.8,9.1,0,2025-05-01,304


In [25]:
# Repeat repair ratio

service_agg["repeat_repair_ratio"] = (
    service_agg["repeat_issue_count"] / service_agg["total_services"]
).replace([np.inf, -np.inf], 0).fillna(0)

service_agg[["vehicle_id", "total_services", "repeat_issue_count", "repeat_repair_ratio"]].head()

,vehicle_id,total_services,repeat_issue_count,repeat_repair_ratio
0,1,9,1,0.111111
1,2,7,1,0.142857
2,3,4,1,0.250000
3,4,3,1,0.333333
4,5,2,0,0.000000


In [26]:
# Severity-level service features

severity_counts = pd.crosstab(service_records["vehicle_id"], service_records["severity"])
severity_counts = severity_counts.reset_index()
severity_counts.head()

severity,vehicle_id,Critical,High,Low,Medium
0,1,0,3,4,2
1,2,0,3,3,1
2,3,0,0,0,4
3,4,0,0,2,1
4,5,0,0,1,1


In [27]:
# Rename severity columns
severity_counts.columns = [str(col).lower().replace(" ", "_") if col != "vehicle_id" else col for col in severity_counts.columns]

severity_rename_map = {
    "low": "low_severity_count",
    "medium": "medium_severity_count",
    "high": "high_severity_count",
    "critical": "critical_severity_count",
    "unknown": "unknown_severity_count"
}

severity_counts = severity_counts.rename(columns=severity_rename_map)
severity_counts.head()

,vehicle_id,critical_severity_count,high_severity_count,low_severity_count,medium_severity_count
0,1,0,3,4,2
1,2,0,3,3,1
2,3,0,0,0,4
3,4,0,0,2,1
4,5,0,0,1,1


In [28]:
# Issue category counts

issue_counts = pd.crosstab(service_records["vehicle_id"], service_records["issue_category"]).reset_index()
issue_counts.columns = [str(col).lower().replace(" ", "_") if col != "vehicle_id" else col for col in issue_counts.columns]

issue_rename_map = {
    "battery": "battery_issue_count",
    "brakes": "brake_issue_count",
    "motor": "motor_issue_count",
    "suspension": "suspension_issue_count",
    "software": "software_issue_count",
    "hvac": "hvac_issue_count",
    "electrical": "electrical_issue_count"
}

issue_counts = issue_counts.rename(columns=issue_rename_map)
issue_counts.head()

,vehicle_id,battery_issue_count,brake_issue_count,electrical_issue_count,hvac_issue_count,motor_issue_count,software_issue_count,suspension_issue_count
0,1,1,0,1,1,1,4,1
1,2,0,0,1,1,2,0,3
2,3,0,0,0,0,1,1,2
3,4,0,0,2,0,0,0,1
4,5,0,0,0,1,0,1,0


In [30]:
# Create telemetry recency helper columns

telemetry_summary["date"] = pd.to_datetime(telemetry_summary["date"])

# Days since telemetry event
telemetry_summary["days_since_telemetry"] = (reference_date - telemetry_summary["date"]).dt.days

# Recency flags
telemetry_summary["telemetry_last_30d"] = (telemetry_summary["days_since_telemetry"] <= 30).astype(int)
telemetry_summary["telemetry_last_90d"] = (telemetry_summary["days_since_telemetry"] <= 90).astype(int)

telemetry_summary.head()

,telemetry_id,vehicle_id,date,battery_health,avg_motor_temp,avg_brake_temp,tire_pressure_alerts,software_version,warning_count,days_since_telemetry,telemetry_last_30d,telemetry_last_90d
0,1,23741,2025-06-27,97.26,77.22,53.09,0,2025.8.1,7,247,0,0
1,2,17245,2025-03-18,98.23,72.30,62.43,2,2025.2.6,1,348,0,0
2,3,34950,2025-04-09,91.76,90.70,56.11,0,2025.2.6,1,326,0,0
3,4,49846,2025-05-31,95.87,81.47,61.18,0,2025.2.6,3,274,0,0
4,5,2253,2025-07-25,82.61,73.74,57.97,0,2025.12.4,2,219,0,0


In [31]:
# Latest telemetry per vehicle

latest_telemetry = telemetry_summary.sort_values(["vehicle_id", "date"]).groupby("vehicle_id").tail(1)

latest_telemetry = latest_telemetry[[
    "vehicle_id",
    "date",
    "battery_health",
    "avg_motor_temp",
    "avg_brake_temp",
    "tire_pressure_alerts",
    "software_version",
    "warning_count"
]].rename(columns={
    "date": "latest_telemetry_date",
    "battery_health": "latest_battery_health",
    "avg_motor_temp": "latest_motor_temp",
    "avg_brake_temp": "latest_brake_temp",
    "tire_pressure_alerts": "latest_tire_pressure_alerts",
    "software_version": "latest_software_version",
    "warning_count": "latest_warning_count"
})

latest_telemetry.head()

,vehicle_id,latest_telemetry_date,latest_battery_health,latest_motor_temp,latest_brake_temp,latest_tire_pressure_alerts,latest_software_version,latest_warning_count
405605,1,2026-02-02,97.86,70.95,47.300000,0,2024.38.1,1
269641,2,2026-02-10,98.84,76.21,54.230000,1,2024.38.1,2
266368,3,2026-02-22,97.26,69.11,69.177330,1,2024.38.1,3
355856,4,2026-02-23,85.54,79.03,69.300000,0,2025.2.6,5
85373,5,2026-02-01,94.90,79.82,66.346378,0,2025.2.6,4


In [32]:
# Aggregate telemetry features

telemetry_agg = telemetry_summary.groupby("vehicle_id").agg(
    telemetry_records=("telemetry_id", "count"),
    avg_battery_health=("battery_health", "mean"),
    min_battery_health=("battery_health", "min"),
    avg_motor_temp=("avg_motor_temp", "mean"),
    max_motor_temp=("avg_motor_temp", "max"),
    avg_brake_temp=("avg_brake_temp", "mean"),
    max_brake_temp=("avg_brake_temp", "max"),
    avg_tire_pressure_alerts=("tire_pressure_alerts", "mean"),
    total_tire_pressure_alerts=("tire_pressure_alerts", "sum"),
    avg_warning_count=("warning_count", "mean"),
    max_warning_count=("warning_count", "max"),
    total_warning_count=("warning_count", "sum"),
    recent_telemetry_30d=("telemetry_last_30d", "sum"),
    recent_telemetry_90d=("telemetry_last_90d", "sum"),
    last_telemetry_date=("date", "max")
).reset_index()

telemetry_agg.head()

,vehicle_id,telemetry_records,avg_battery_health,min_battery_health,avg_motor_temp,max_motor_temp,avg_brake_temp,max_brake_temp,avg_tire_pressure_alerts,total_tire_pressure_alerts,avg_warning_count,max_warning_count,total_warning_count,recent_telemetry_30d,recent_telemetry_90d,last_telemetry_date
0,1,8,97.595000,95.70,70.687500,85.57,57.071250,69.930000,0.250000,2,1.500000,3,12,1,2,2026-02-02
1,2,15,97.442667,92.61,70.212667,96.13,60.308000,70.240000,0.466667,7,1.600000,4,24,1,6,2026-02-10
2,3,6,92.931667,89.57,66.558333,75.42,59.407898,72.692421,0.333333,2,1.833333,3,11,2,3,2026-02-22
3,4,14,81.323571,75.99,70.904286,90.87,58.517143,69.300000,0.857143,12,2.785714,7,39,1,3,2026-02-23
4,5,11,93.283636,89.25,74.371818,94.77,61.857375,68.134728,0.454545,5,2.090909,4,23,1,1,2026-02-01


In [33]:
# Days since last telemetry
telemetry_agg["days_since_last_telemetry"] = (reference_date - telemetry_agg["last_telemetry_date"]).dt.days
telemetry_agg.head()

,vehicle_id,telemetry_records,avg_battery_health,min_battery_health,avg_motor_temp,max_motor_temp,avg_brake_temp,max_brake_temp,avg_tire_pressure_alerts,total_tire_pressure_alerts,avg_warning_count,max_warning_count,total_warning_count,recent_telemetry_30d,recent_telemetry_90d,last_telemetry_date,days_since_last_telemetry
0,1,8,97.595000,95.70,70.687500,85.57,57.071250,69.930000,0.250000,2,1.500000,3,12,1,2,2026-02-02,27
1,2,15,97.442667,92.61,70.212667,96.13,60.308000,70.240000,0.466667,7,1.600000,4,24,1,6,2026-02-10,19
2,3,6,92.931667,89.57,66.558333,75.42,59.407898,72.692421,0.333333,2,1.833333,3,11,2,3,2026-02-22,7
3,4,14,81.323571,75.99,70.904286,90.87,58.517143,69.300000,0.857143,12,2.785714,7,39,1,3,2026-02-23,6
4,5,11,93.283636,89.25,74.371818,94.77,61.857375,68.134728,0.454545,5,2.090909,4,23,1,1,2026-02-01,28


In [35]:
# Warranty recency helper
warranty_claims["claim_date"] = pd.to_datetime(warranty_claims["claim_date"])

# Calculate recency
warranty_claims["days_since_claim"] = (reference_date - warranty_claims["claim_date"]).dt.days

# Create recency flags
warranty_claims["claim_last_90d"] = (warranty_claims["days_since_claim"] <= 90).astype(int)
warranty_claims["claim_last_180d"] = (warranty_claims["days_since_claim"] <= 180).astype(int)

warranty_claims.head()

,claim_id,vehicle_id,claim_date,component,claim_amount,claim_status,days_since_claim,claim_last_90d,claim_last_180d
0,1,28788,2022-08-11,Wiring Harness,2325.68,Pending,1298,0,0
1,2,46239,2023-11-26,BMS,7554.90,Approved,826,0,0
2,3,11206,2022-08-05,Drive Unit,5738.37,Pending,1304,0,0
3,4,23091,2022-01-30,Drive Unit,7381.06,Approved,1491,0,0
4,5,521,2023-11-23,Cabin Heater,2123.84,Approved,829,0,0


In [36]:
# Aggregate warranty features

warranty_agg = warranty_claims.groupby("vehicle_id").agg(
    total_claims=("claim_id", "count"),
    claims_last_90d=("claim_last_90d", "sum"),
    claims_last_180d=("claim_last_180d", "sum"),
    total_claim_amount=("claim_amount", "sum"),
    avg_claim_amount=("claim_amount", "mean"),
    max_claim_amount=("claim_amount", "max"),
    last_claim_date=("claim_date", "max")
).reset_index()

warranty_agg.head()

,vehicle_id,total_claims,claims_last_90d,claims_last_180d,total_claim_amount,avg_claim_amount,max_claim_amount,last_claim_date
0,1,3,0,1,3621.34,1207.113333,1553.92,2025-10-14
1,2,3,0,0,4276.06,1425.353333,2354.44,2024-04-13
2,3,1,0,0,676.66,676.660000,676.66,2023-05-31
3,4,2,0,0,4122.56,2061.280000,2747.44,2023-01-20
4,5,2,1,1,9909.76,4954.880000,6986.76,2026-01-09


In [37]:
# Days since last claim

warranty_agg["days_since_last_claim"] = (reference_date - warranty_agg["last_claim_date"]).dt.days
warranty_agg.head()

,vehicle_id,total_claims,claims_last_90d,claims_last_180d,total_claim_amount,avg_claim_amount,max_claim_amount,last_claim_date,days_since_last_claim
0,1,3,0,1,3621.34,1207.113333,1553.92,2025-10-14,138
1,2,3,0,0,4276.06,1425.353333,2354.44,2024-04-13,687
2,3,1,0,0,676.66,676.660000,676.66,2023-05-31,1005
3,4,2,0,0,4122.56,2061.280000,2747.44,2023-01-20,1136
4,5,2,1,1,9909.76,4954.880000,6986.76,2026-01-09,51


In [38]:
# Claim status counts

claim_status_counts = pd.crosstab(warranty_claims["vehicle_id"], warranty_claims["claim_status"]).reset_index()
claim_status_counts.columns = [str(col).lower().replace(" ", "_") if col != "vehicle_id" else col for col in claim_status_counts.columns]

claim_status_rename_map = {
    "approved": "approved_claim_count",
    "pending": "pending_claim_count",
    "rejected": "rejected_claim_count"
}

claim_status_counts = claim_status_counts.rename(columns=claim_status_rename_map)
claim_status_counts.head()

,vehicle_id,approved_claim_count,pending_claim_count,rejected_claim_count
0,1,0,3,0
1,2,2,0,1
2,3,0,1,0
3,4,0,0,2
4,5,1,1,0


In [40]:
# Failure recency helper

parts_failure["failure_date"] = pd.to_datetime(parts_failure["failure_date"])

# Calculate days since failure
parts_failure["days_since_failure"] = (reference_date - parts_failure["failure_date"]).dt.days

# Recency flags
parts_failure["failure_last_90d"] = (parts_failure["days_since_failure"] <= 90).astype(int)
parts_failure["failure_last_180d"] = (parts_failure["days_since_failure"] <= 180).astype(int)

parts_failure.head()

,failure_id,vehicle_id,component,failure_date,failure_type,replaced_flag,days_since_failure,failure_last_90d,failure_last_180d
0,1,5821,Cabin Heater,2025-08-25,Thermal Issue,1,188,0,0
1,2,13420,ABS Module,2025-06-16,Thermal Issue,1,258,0,0
2,3,44,Cooling Fan,2026-02-09,Software Error,1,20,1,1
3,4,27146,Motor Controller,2021-03-04,Software Error,0,1823,0,0
4,5,48454,Wiring Harness,2021-02-05,Thermal Issue,0,1850,0,0


In [41]:
# Aggregate warranty features
warranty_agg = warranty_claims.groupby("vehicle_id").agg(
    total_claims=("claim_id", "count"),
    claims_last_90d=("claim_last_90d", "sum"),
    claims_last_180d=("claim_last_180d", "sum"),
    total_claim_amount=("claim_amount", "sum"),
    avg_claim_amount=("claim_amount", "mean"),
    max_claim_amount=("claim_amount", "max"),
    last_claim_date=("claim_date", "max")
).reset_index()

warranty_agg.head()


,vehicle_id,total_claims,claims_last_90d,claims_last_180d,total_claim_amount,avg_claim_amount,max_claim_amount,last_claim_date
0,1,3,0,1,3621.34,1207.113333,1553.92,2025-10-14
1,2,3,0,0,4276.06,1425.353333,2354.44,2024-04-13
2,3,1,0,0,676.66,676.660000,676.66,2023-05-31
3,4,2,0,0,4122.56,2061.280000,2747.44,2023-01-20
4,5,2,1,1,9909.76,4954.880000,6986.76,2026-01-09


In [42]:
# Days since last claim
warranty_agg["days_since_last_claim"] = (reference_date - warranty_agg["last_claim_date"]).dt.days
warranty_agg.head()

,vehicle_id,total_claims,claims_last_90d,claims_last_180d,total_claim_amount,avg_claim_amount,max_claim_amount,last_claim_date,days_since_last_claim
0,1,3,0,1,3621.34,1207.113333,1553.92,2025-10-14,138
1,2,3,0,0,4276.06,1425.353333,2354.44,2024-04-13,687
2,3,1,0,0,676.66,676.660000,676.66,2023-05-31,1005
3,4,2,0,0,4122.56,2061.280000,2747.44,2023-01-20,1136
4,5,2,1,1,9909.76,4954.880000,6986.76,2026-01-09,51


In [43]:
# Claim status counts

claim_status_counts = pd.crosstab(warranty_claims["vehicle_id"], warranty_claims["claim_status"]).reset_index()
claim_status_counts.columns = [str(col).lower().replace(" ", "_") if col != "vehicle_id" else col for col in claim_status_counts.columns]

claim_status_rename_map = {
    "approved": "approved_claim_count",
    "pending": "pending_claim_count",
    "rejected": "rejected_claim_count"
}

claim_status_counts = claim_status_counts.rename(columns=claim_status_rename_map)
claim_status_counts.head()

,vehicle_id,approved_claim_count,pending_claim_count,rejected_claim_count
0,1,0,3,0
1,2,2,0,1
2,3,0,1,0
3,4,0,0,2
4,5,1,1,0


In [45]:
# Failure recency helper

parts_failure["failure_date"] = pd.to_datetime(parts_failure["failure_date"])

# Calculate recency
parts_failure["days_since_failure"] = (reference_date - parts_failure["failure_date"]).dt.days

# Create recency flags
parts_failure["failure_last_90d"] = (parts_failure["days_since_failure"] <= 90).astype(int)
parts_failure["failure_last_180d"] = (parts_failure["days_since_failure"] <= 180).astype(int)

parts_failure.head()

,failure_id,vehicle_id,component,failure_date,failure_type,replaced_flag,days_since_failure,failure_last_90d,failure_last_180d
0,1,5821,Cabin Heater,2025-08-25,Thermal Issue,1,188,0,0
1,2,13420,ABS Module,2025-06-16,Thermal Issue,1,258,0,0
2,3,44,Cooling Fan,2026-02-09,Software Error,1,20,1,1
3,4,27146,Motor Controller,2021-03-04,Software Error,0,1823,0,0
4,5,48454,Wiring Harness,2021-02-05,Thermal Issue,0,1850,0,0


In [46]:
# Aggregate parts failure features

failure_agg = parts_failure.groupby("vehicle_id").agg(
    total_failures=("failure_id", "count"),
    failures_last_90d=("failure_last_90d", "sum"),
    failures_last_180d=("failure_last_180d", "sum"),
    replaced_count=("replaced_flag", "sum"),
    last_failure_date=("failure_date", "max")
).reset_index()

failure_agg.head()

,vehicle_id,total_failures,failures_last_90d,failures_last_180d,replaced_count,last_failure_date
0,1,3,0,0,2,2024-06-27
1,2,3,0,0,2,2024-07-29
2,3,7,0,0,7,2025-02-18
3,4,2,0,0,2,2024-12-04
4,5,4,0,0,3,2025-06-18


In [47]:
# Days since last failure
failure_agg["days_since_last_failure"] = (reference_date - failure_agg["last_failure_date"]).dt.days
failure_agg.head()

,vehicle_id,total_failures,failures_last_90d,failures_last_180d,replaced_count,last_failure_date,days_since_last_failure
0,1,3,0,0,2,2024-06-27,612
1,2,3,0,0,2,2024-07-29,580
2,3,7,0,0,7,2025-02-18,376
3,4,2,0,0,2,2024-12-04,452
4,5,4,0,0,3,2025-06-18,256


In [48]:
# Failure Type Counts
failure_type_counts = pd.crosstab(parts_failure["vehicle_id"], parts_failure["failure_type"]).reset_index()
failure_type_counts.columns = [str(col).lower().replace(" ", "_") if col != "vehicle_id" else col for col in failure_type_counts.columns]

failure_type_counts.head()

,vehicle_id,electrical_fault,mechanical_failure,software_error,thermal_issue,wear
0,1,0,0,1,1,1
1,2,0,0,1,1,1
2,3,1,0,3,1,2
3,4,0,0,0,1,1
4,5,2,1,0,0,1


In [49]:
# Component Failure Counts

component_failure_counts = pd.crosstab(parts_failure["vehicle_id"], parts_failure["component"]).reset_index()
component_failure_counts.columns = [str(col).lower().replace(" ", "_") if col != "vehicle_id" else col for col in component_failure_counts.columns]

component_failure_counts.head()

,vehicle_id,12v_battery,abs_module,ac_compressor,autopilot_computer,bms,battery_pack,brake_pads,brake_rotor,cabin_heater,charging_port,control_arm,cooling_fan,drive_unit,firmware,infotainment_module,inverter,motor_controller,sensor_module,shock_absorber,suspension_link,wiring_harness
0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0
1,2,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
2,3,1,0,0,0,1,1,1,0,0,0,0,0,0,0,0,1,1,0,0,0,1
3,4,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0
4,5,2,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [50]:
# Merge service features

vehicle_features = vehicle_features.merge(service_agg, on="vehicle_id", how="left")
vehicle_features = vehicle_features.merge(severity_counts, on="vehicle_id", how="left")
vehicle_features = vehicle_features.merge(issue_counts, on="vehicle_id", how="left")

print(vehicle_features.shape)
vehicle_features.head()

(50000, 38)


,vehicle_id,vin,model,model_year,battery_type,delivery_date,region,mileage,fleet_type,vehicle_age_days,vehicle_age_months,vehicle_age_years,mileage_band,total_services,repairs_last_30d,repairs_last_90d,repairs_last_180d,avg_repair_cost,max_repair_cost,total_repair_cost,avg_downtime_days,max_downtime_days,total_downtime_days,repeat_issue_count,last_service_date,days_since_last_service,repeat_repair_ratio,critical_severity_count,high_severity_count,low_severity_count,medium_severity_count,battery_issue_count,brake_issue_count,electrical_issue_count,hvac_issue_count,motor_issue_count,software_issue_count,suspension_issue_count
0,1,5YJ00000000000001,Model 3,2025,NCA,2020-04-11,Asia Pacific,1269,Retail,2150,71.7,5.89,Low,9.0,0.0,1.0,1.0,1641.293333,5206.91,14771.64,2.633333,5.4,23.7,1.0,2025-12-14,77.0,0.111111,0.0,3.0,4.0,2.0,1.0,0.0,1.0,1.0,1.0,4.0,1.0
1,2,5YJ00000000000002,Model Y,2025,NCM,2019-01-22,North America,3505,Retail,2595,86.5,7.11,Low,7.0,0.0,1.0,2.0,1715.110000,3449.08,12005.77,2.457143,4.0,17.2,1.0,2026-01-01,59.0,0.142857,0.0,3.0,3.0,1.0,0.0,0.0,1.0,1.0,2.0,0.0,3.0
2,3,5YJ00000000000003,Model 3,2023,LFP,2024-08-30,Europe,58323,Retail,548,18.3,1.50,High,4.0,0.0,0.0,2.0,1585.157500,2295.66,6340.63,4.000000,6.4,16.0,1.0,2025-11-20,101.0,0.250000,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0
3,4,5YJ00000000000004,Model 3,2019,LFP,2025-12-31,North America,87159,Retail,60,2.0,0.16,High,3.0,0.0,0.0,0.0,1760.020000,3445.09,5280.06,4.366667,5.1,13.1,1.0,2024-12-25,431.0,0.333333,0.0,0.0,2.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,1.0
4,5,5YJ00000000000005,Model 3,2023,LFP,2023-04-29,Europe,44127,Internal Test,1037,34.6,2.84,Moderate,2.0,0.0,0.0,0.0,721.215000,1353.66,1442.43,4.550000,4.8,9.1,0.0,2025-05-01,304.0,0.000000,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [51]:
# Merge telemetry features

vehicle_features = vehicle_features.merge(telemetry_agg, on="vehicle_id", how="left")
vehicle_features = vehicle_features.merge(latest_telemetry, on="vehicle_id", how="left")

print(vehicle_features.shape)
vehicle_features.head()

(50000, 61)


,vehicle_id,vin,model,model_year,battery_type,delivery_date,region,mileage,fleet_type,vehicle_age_days,vehicle_age_months,vehicle_age_years,mileage_band,total_services,repairs_last_30d,repairs_last_90d,repairs_last_180d,avg_repair_cost,max_repair_cost,total_repair_cost,avg_downtime_days,max_downtime_days,total_downtime_days,repeat_issue_count,last_service_date,days_since_last_service,repeat_repair_ratio,critical_severity_count,high_severity_count,low_severity_count,medium_severity_count,battery_issue_count,brake_issue_count,electrical_issue_count,hvac_issue_count,motor_issue_count,software_issue_count,suspension_issue_count,telemetry_records,avg_battery_health,min_battery_health,avg_motor_temp,max_motor_temp,avg_brake_temp,max_brake_temp,avg_tire_pressure_alerts,total_tire_pressure_alerts,avg_warning_count,max_warning_count,total_warning_count,recent_telemetry_30d,recent_telemetry_90d,last_telemetry_date,days_since_last_telemetry,latest_telemetry_date,latest_battery_health,latest_motor_temp,latest_brake_temp,latest_tire_pressure_alerts,latest_software_version,latest_warning_count
0,1,5YJ00000000000001,Model 3,2025,NCA,2020-04-11,Asia Pacific,1269,Retail,2150,71.7,5.89,Low,9.0,0.0,1.0,1.0,1641.293333,5206.91,14771.64,2.633333,5.4,23.7,1.0,2025-12-14,77.0,0.111111,0.0,3.0,4.0,2.0,1.0,0.0,1.0,1.0,1.0,4.0,1.0,8.0,97.595000,95.70,70.687500,85.57,57.071250,69.930000,0.250000,2.0,1.500000,3.0,12.0,1.0,2.0,2026-02-02,27.0,2026-02-02,97.86,70.95,47.300000,0.0,2024.38.1,1.0
1,2,5YJ00000000000002,Model Y,2025,NCM,2019-01-22,North America,3505,Retail,2595,86.5,7.11,Low,7.0,0.0,1.0,2.0,1715.110000,3449.08,12005.77,2.457143,4.0,17.2,1.0,2026-01-01,59.0,0.142857,0.0,3.0,3.0,1.0,0.0,0.0,1.0,1.0,2.0,0.0,3.0,15.0,97.442667,92.61,70.212667,96.13,60.308000,70.240000,0.466667,7.0,1.600000,4.0,24.0,1.0,6.0,2026-02-10,19.0,2026-02-10,98.84,76.21,54.230000,1.0,2024.38.1,2.0
2,3,5YJ00000000000003,Model 3,2023,LFP,2024-08-30,Europe,58323,Retail,548,18.3,1.50,High,4.0,0.0,0.0,2.0,1585.157500,2295.66,6340.63,4.000000,6.4,16.0,1.0,2025-11-20,101.0,0.250000,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,6.0,92.931667,89.57,66.558333,75.42,59.407898,72.692421,0.333333,2.0,1.833333,3.0,11.0,2.0,3.0,2026-02-22,7.0,2026-02-22,97.26,69.11,69.177330,1.0,2024.38.1,3.0
3,4,5YJ00000000000004,Model 3,2019,LFP,2025-12-31,North America,87159,Retail,60,2.0,0.16,High,3.0,0.0,0.0,0.0,1760.020000,3445.09,5280.06,4.366667,5.1,13.1,1.0,2024-12-25,431.0,0.333333,0.0,0.0,2.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,1.0,14.0,81.323571,75.99,70.904286,90.87,58.517143,69.300000,0.857143,12.0,2.785714,7.0,39.0,1.0,3.0,2026-02-23,6.0,2026-02-23,85.54,79.03,69.300000,0.0,2025.2.6,5.0
4,5,5YJ00000000000005,Model 3,2023,LFP,2023-04-29,Europe,44127,Internal Test,1037,34.6,2.84,Moderate,2.0,0.0,0.0,0.0,721.215000,1353.66,1442.43,4.550000,4.8,9.1,0.0,2025-05-01,304.0,0.000000,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,11.0,93.283636,89.25,74.371818,94.77,61.857375,68.134728,0.454545,5.0,2.090909,4.0,23.0,1.0,1.0,2026-02-01,28.0,2026-02-01,94.90,79.82,66.346378,0.0,2025.2.6,4.0


In [52]:
# Merge warranty features

vehicle_features = vehicle_features.merge(warranty_agg, on="vehicle_id", how="left")
vehicle_features = vehicle_features.merge(claim_status_counts, on="vehicle_id", how="left")

print(vehicle_features.shape)
vehicle_features.head()

(50000, 72)


,vehicle_id,vin,model,model_year,battery_type,delivery_date,region,mileage,fleet_type,vehicle_age_days,vehicle_age_months,vehicle_age_years,mileage_band,total_services,repairs_last_30d,repairs_last_90d,repairs_last_180d,avg_repair_cost,max_repair_cost,total_repair_cost,avg_downtime_days,max_downtime_days,total_downtime_days,repeat_issue_count,last_service_date,days_since_last_service,repeat_repair_ratio,critical_severity_count,high_severity_count,low_severity_count,medium_severity_count,battery_issue_count,brake_issue_count,electrical_issue_count,hvac_issue_count,motor_issue_count,software_issue_count,suspension_issue_count,telemetry_records,avg_battery_health,min_battery_health,avg_motor_temp,max_motor_temp,avg_brake_temp,max_brake_temp,avg_tire_pressure_alerts,total_tire_pressure_alerts,avg_warning_count,max_warning_count,total_warning_count,recent_telemetry_30d,recent_telemetry_90d,last_telemetry_date,days_since_last_telemetry,latest_telemetry_date,latest_battery_health,latest_motor_temp,latest_brake_temp,latest_tire_pressure_alerts,latest_software_version,latest_warning_count,total_claims,claims_last_90d,claims_last_180d,total_claim_amount,avg_claim_amount,max_claim_amount,last_claim_date,days_since_last_claim,approved_claim_count,pending_claim_count,rejected_claim_count
0,1,5YJ00000000000001,Model 3,2025,NCA,2020-04-11,Asia Pacific,1269,Retail,2150,71.7,5.89,Low,9.0,0.0,1.0,1.0,1641.293333,5206.91,14771.64,2.633333,5.4,23.7,1.0,2025-12-14,77.0,0.111111,0.0,3.0,4.0,2.0,1.0,0.0,1.0,1.0,1.0,4.0,1.0,8.0,97.595000,95.70,70.687500,85.57,57.071250,69.930000,0.250000,2.0,1.500000,3.0,12.0,1.0,2.0,2026-02-02,27.0,2026-02-02,97.86,70.95,47.300000,0.0,2024.38.1,1.0,3.0,0.0,1.0,3621.34,1207.113333,1553.92,2025-10-14,138.0,0.0,3.0,0.0
1,2,5YJ00000000000002,Model Y,2025,NCM,2019-01-22,North America,3505,Retail,2595,86.5,7.11,Low,7.0,0.0,1.0,2.0,1715.110000,3449.08,12005.77,2.457143,4.0,17.2,1.0,2026-01-01,59.0,0.142857,0.0,3.0,3.0,1.0,0.0,0.0,1.0,1.0,2.0,0.0,3.0,15.0,97.442667,92.61,70.212667,96.13,60.308000,70.240000,0.466667,7.0,1.600000,4.0,24.0,1.0,6.0,2026-02-10,19.0,2026-02-10,98.84,76.21,54.230000,1.0,2024.38.1,2.0,3.0,0.0,0.0,4276.06,1425.353333,2354.44,2024-04-13,687.0,2.0,0.0,1.0
2,3,5YJ00000000000003,Model 3,2023,LFP,2024-08-30,Europe,58323,Retail,548,18.3,1.50,High,4.0,0.0,0.0,2.0,1585.157500,2295.66,6340.63,4.000000,6.4,16.0,1.0,2025-11-20,101.0,0.250000,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,6.0,92.931667,89.57,66.558333,75.42,59.407898,72.692421,0.333333,2.0,1.833333,3.0,11.0,2.0,3.0,2026-02-22,7.0,2026-02-22,97.26,69.11,69.177330,1.0,2024.38.1,3.0,1.0,0.0,0.0,676.66,676.660000,676.66,2023-05-31,1005.0,0.0,1.0,0.0
3,4,5YJ00000000000004,Model 3,2019,LFP,2025-12-31,North America,87159,Retail,60,2.0,0.16,High,3.0,0.0,0.0,0.0,1760.020000,3445.09,5280.06,4.366667,5.1,13.1,1.0,2024-12-25,431.0,0.333333,0.0,0.0,2.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,1.0,14.0,81.323571,75.99,70.904286,90.87,58.517143,69.300000,0.857143,12.0,2.785714,7.0,39.0,1.0,3.0,2026-02-23,6.0,2026-02-23,85.54,79.03,69.300000,0.0,2025.2.6,5.0,2.0,0.0,0.0,4122.56,2061.280000,2747.44,2023-01-20,1136.0,0.0,0.0,2.0
4,5,5YJ00000000000005,Model 3,2023,LFP,2023-04-29,Europe,44127,Internal Test,1037,34.6,2.84,Moderate,2.0,0.0,0.0,0.0,721.215000,1353.66,1442.43,4.550000,4.8,9.1,0.0,2025-05-01,304.0,0.000000,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,11.0,93.283636,89.25,74.371818,94.77,61.857375,68.134728,0.454545,5.0,2.090909,4.0,23.0,1.0,1.0,2026-02-01,28.0,2026-02-01,94.90,79.82,66.346378,0.0,2025.2.6,4.0,2.0,1.0,1.0,9909.76,4954.880000,6986.76,2026-01-09,51.0,1.0,1.0,0.0


In [53]:
# Merge failure features

vehicle_features = vehicle_features.merge(failure_agg, on="vehicle_id", how="left")
vehicle_features = vehicle_features.merge(failure_type_counts, on="vehicle_id", how="left")
vehicle_features = vehicle_features.merge(component_failure_counts, on="vehicle_id", how="left")

print(vehicle_features.shape)
vehicle_features.head()

(50000, 104)


,vehicle_id,vin,model,model_year,battery_type,delivery_date,region,mileage,fleet_type,vehicle_age_days,vehicle_age_months,vehicle_age_years,mileage_band,total_services,repairs_last_30d,repairs_last_90d,repairs_last_180d,avg_repair_cost,max_repair_cost,total_repair_cost,avg_downtime_days,max_downtime_days,total_downtime_days,repeat_issue_count,last_service_date,days_since_last_service,repeat_repair_ratio,critical_severity_count,high_severity_count,low_severity_count,medium_severity_count,battery_issue_count,brake_issue_count,electrical_issue_count,hvac_issue_count,motor_issue_count,software_issue_count,suspension_issue_count,telemetry_records,avg_battery_health,min_battery_health,avg_motor_temp,max_motor_temp,avg_brake_temp,max_brake_temp,avg_tire_pressure_alerts,total_tire_pressure_alerts,avg_warning_count,max_warning_count,total_warning_count,recent_telemetry_30d,recent_telemetry_90d,last_telemetry_date,days_since_last_telemetry,latest_telemetry_date,latest_battery_health,latest_motor_temp,latest_brake_temp,latest_tire_pressure_alerts,latest_software_version,latest_warning_count,total_claims,claims_last_90d,claims_last_180d,total_claim_amount,avg_claim_amount,max_claim_amount,last_claim_date,days_since_last_claim,approved_claim_count,pending_claim_count,rejected_claim_count,total_failures,failures_last_90d,failures_last_180d,replaced_count,last_failure_date,days_since_last_failure,electrical_fault,mechanical_failure,software_error,thermal_issue,wear,12v_battery,abs_module,ac_compressor,autopilot_computer,bms,battery_pack,brake_pads,brake_rotor,cabin_heater,charging_port,control_arm,cooling_fan,drive_unit,firmware,infotainment_module,inverter,motor_controller,sensor_module,shock_absorber,suspension_link,wiring_harness
0,1,5YJ00000000000001,Model 3,2025,NCA,2020-04-11,Asia Pacific,1269,Retail,2150,71.7,5.89,Low,9.0,0.0,1.0,1.0,1641.293333,5206.91,14771.64,2.633333,5.4,23.7,1.0,2025-12-14,77.0,0.111111,0.0,3.0,4.0,2.0,1.0,0.0,1.0,1.0,1.0,4.0,1.0,8.0,97.595000,95.70,70.687500,85.57,57.071250,69.930000,0.250000,2.0,1.500000,3.0,12.0,1.0,2.0,2026-02-02,27.0,2026-02-02,97.86,70.95,47.300000,0.0,2024.38.1,1.0,3.0,0.0,1.0,3621.34,1207.113333,1553.92,2025-10-14,138.0,0.0,3.0,0.0,3.0,0.0,0.0,2.0,2024-06-27,612.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,5YJ00000000000002,Model Y,2025,NCM,2019-01-22,North America,3505,Retail,2595,86.5,7.11,Low,7.0,0.0,1.0,2.0,1715.110000,3449.08,12005.77,2.457143,4.0,17.2,1.0,2026-01-01,59.0,0.142857,0.0,3.0,3.0,1.0,0.0,0.0,1.0,1.0,2.0,0.0,3.0,15.0,97.442667,92.61,70.212667,96.13,60.308000,70.240000,0.466667,7.0,1.600000,4.0,24.0,1.0,6.0,2026-02-10,19.0,2026-02-10,98.84,76.21,54.230000,1.0,2024.38.1,2.0,3.0,0.0,0.0,4276.06,1425.353333,2354.44,2024-04-13,687.0,2.0,0.0,1.0,3.0,0.0,0.0,2.0,2024-07-29,580.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,5YJ00000000000003,Model 3,2023,LFP,2024-08-30,Europe,58323,Retail,548,18.3,1.50,High,4.0,0.0,0.0,2.0,1585.157500,2295.66,6340.63,4.000000,6.4,16.0,1.0,2025-11-20,101.0,0.250000,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,6.0,92.931667,89.57,66.558333,75.42,59.407898,72.692421,0.333333,2.0,1.833333,3.0,11.0,2.0,3.0,2026-02-22,7.0,2026-02-22,97.26,69.11,69.177330,1.0,2024.38.1,3.0,1.0,0.0,0.0,676.66,676.660000,676.66,2023-05-31,1005.0,0.0,1.0,0.0,7.0,0.0,0.0,7.0,2025-02-18,376.0,1.0,0.0,3.0,1.0,2.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
3,4,5YJ00000000000004,Model 3,2019,LFP,2025-12-31,North America,87159,Retail,60,2.0,0.16,High,3.0,0.0,0.0,0.0,1760.020000,3445.09,5280.06,4.366667,5.1,13.1,1.0,2024-12-25,431.0,0.333333,0.0,0.0,2.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,1.0,14.0,81.323571,75.99,70.904286,90.87,58.517143,69.300000,0.857143,12.0,2.785714,7.0,39.0,1.0,3.0,2026-02-23,6.0,2026-02-23,85.54,79.03,69.300000,0.0,2025.2.6,5.0,2.0,0.0,0.0,4122.56,2061.280000,2747.44,2023-01-20,1136.0,0.0,0.0,2.0,2.

In [54]:
# Fill numeric nulls after merge

numeric_cols = vehicle_features.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    vehicle_features[col] = vehicle_features[col].fillna(0)

print("Numeric nulls filled.")

Numeric nulls filled.


In [55]:
# Fill selected text nulls

object_cols = vehicle_features.select_dtypes(include=["object"]).columns

for col in object_cols:
    vehicle_features[col] = vehicle_features[col].fillna("Unknown")

print("Object nulls filled.")

Object nulls filled.


In [56]:
# Fill datetime nulls carefully

datetime_cols = vehicle_features.select_dtypes(include=["datetime64[ns]"]).columns
print("Datetime columns:", list(datetime_cols))

Datetime columns: ['delivery_date', 'last_service_date', 'last_telemetry_date', 'latest_telemetry_date', 'last_claim_date', 'last_failure_date']


In [57]:
# Derived ratios and indicators

vehicle_features["claims_per_service"] = (
    vehicle_features["total_claims"] / vehicle_features["total_services"]
).replace([np.inf, -np.inf], 0).fillna(0)

vehicle_features["failures_per_service"] = (
    vehicle_features["total_failures"] / vehicle_features["total_services"]
).replace([np.inf, -np.inf], 0).fillna(0)

vehicle_features["avg_cost_per_downtime_day"] = (
    vehicle_features["total_repair_cost"] / vehicle_features["total_downtime_days"]
).replace([np.inf, -np.inf], 0).fillna(0)

vehicle_features["warning_to_failure_ratio"] = (
    vehicle_features["total_warning_count"] / vehicle_features["total_failures"]
).replace([np.inf, -np.inf], 0).fillna(0)

vehicle_features["service_per_year"] = (
    vehicle_features["total_services"] / vehicle_features["vehicle_age_years"]
).replace([np.inf, -np.inf], 0).fillna(0)

vehicle_features["claim_per_year"] = (
    vehicle_features["total_claims"] / vehicle_features["vehicle_age_years"]
).replace([np.inf, -np.inf], 0).fillna(0)

vehicle_features["failure_per_year"] = (
    vehicle_features["total_failures"] / vehicle_features["vehicle_age_years"]
).replace([np.inf, -np.inf], 0).fillna(0)

vehicle_features.head()

/tmp/ipykernel_174/3616373471.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  vehicle_features["failures_per_service"] = (
/tmp/ipykernel_174/3616373471.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  vehicle_features["avg_cost_per_downtime_day"] = (
/tmp/ipykernel_174/3616373471.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragment

,vehicle_id,vin,model,model_year,battery_type,delivery_date,region,mileage,fleet_type,vehicle_age_days,vehicle_age_months,vehicle_age_years,mileage_band,total_services,repairs_last_30d,repairs_last_90d,repairs_last_180d,avg_repair_cost,max_repair_cost,total_repair_cost,avg_downtime_days,max_downtime_days,total_downtime_days,repeat_issue_count,last_service_date,days_since_last_service,repeat_repair_ratio,critical_severity_count,high_severity_count,low_severity_count,medium_severity_count,battery_issue_count,brake_issue_count,electrical_issue_count,hvac_issue_count,motor_issue_count,software_issue_count,suspension_issue_count,telemetry_records,avg_battery_health,min_battery_health,avg_motor_temp,max_motor_temp,avg_brake_temp,max_brake_temp,avg_tire_pressure_alerts,total_tire_pressure_alerts,avg_warning_count,max_warning_count,total_warning_count,recent_telemetry_30d,recent_telemetry_90d,last_telemetry_date,days_since_last_telemetry,latest_telemetry_date,latest_battery_health,latest_motor_temp,latest_brake_temp,latest_tire_pressure_alerts,latest_software_version,latest_warning_count,total_claims,claims_last_90d,claims_last_180d,total_claim_amount,avg_claim_amount,max_claim_amount,last_claim_date,days_since_last_claim,approved_claim_count,pending_claim_count,rejected_claim_count,total_failures,failures_last_90d,failures_last_180d,replaced_count,last_failure_date,days_since_last_failure,electrical_fault,mechanical_failure,software_error,thermal_issue,wear,12v_battery,abs_module,ac_compressor,autopilot_computer,bms,battery_pack,brake_pads,brake_rotor,cabin_heater,charging_port,control_arm,cooling_fan,drive_unit,firmware,infotainment_module,inverter,motor_controller,sensor_module,shock_absorber,suspension_link,wiring_harness,claims_per_service,failures_per_service,avg_cost_per_downtime_day,warning_to_failure_ratio,service_per_year,claim_per_year,failure_per_year
0,1,5YJ00000000000001,Model 3,2025,NCA,2020-04-11,Asia Pacific,1269,Retail,2150,71.7,5.89,Low,9.0,0.0,1.0,1.0,1641.293333,5206.91,14771.64,2.633333,5.4,23.7,1.0,2025-12-14,77.0,0.111111,0.0,3.0,4.0,2.0,1.0,0.0,1.0,1.0,1.0,4.0,1.0,8.0,97.595000,95.70,70.687500,85.57,57.071250,69.930000,0.250000,2.0,1.500000,3.0,12.0,1.0,2.0,2026-02-02,27.0,2026-02-02,97.86,70.95,47.300000,0.0,2024.38.1,1.0,3.0,0.0,1.0,3621.34,1207.113333,1553.92,2025-10-14,138.0,0.0,3.0,0.0,3.0,0.0,0.0,2.0,2024-06-27,612.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,0.333333,623.275949,4.000000,1.528014,0.509338,0.509338
1,2,5YJ00000000000002,Model Y,2025,NCM,2019-01-22,North America,3505,Retail,2595,86.5,7.11,Low,7.0,0.0,1.0,2.0,1715.110000,3449.08,12005.77,2.457143,4.0,17.2,1.0,2026-01-01,59.0,0.142857,0.0,3.0,3.0,1.0,0.0,0.0,1.0,1.0,2.0,0.0,3.0,15.0,97.442667,92.61,70.212667,96.13,60.308000,70.240000,0.466667,7.0,1.600000,4.0,24.0,1.0,6.0,2026-02-10,19.0,2026-02-10,98.84,76.21,54.230000,1.0,2024.38.1,2.0,3.0,0.0,0.0,4276.06,1425.353333,2354.44,2024-04-13,687.0,2.0,0.0,1.0,3.0,0.0,0.0,2.0,2024-07-29,580.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.428571,0.428571,698.009884,8.000000,0.984529,0.421941,0.421941
2,3,5YJ00000000000003,Model 3,2023,LFP,2024-08-30,Europe,58323,Retail,548,18.3,1.50,High,4.0,0.0,0.0,2.0,1585.157500,2295.66,6340.63,4.000000,6.4,16.0,1.0,2025-11-20,101.0,0.250000,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,6.0,92.931667,89.57,66.558333,75.42,59.407898,72.692421,0.333333,2.0,1.833333,3.0,11.0,2.0,3.0,2026-02-22,7.0,2026-02-22,97.26,69.11,69.177330,1.0,2024.38.1,3.0,1.0,0.0,0.0,676.66,676.660000,676.66,2023-05-31,1005.0,0.0,1.0,0.0,7.0,0.0,0.0,7.0,2025-02-18,376.0,1.0,0.0,3.0,1.0,2.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.250000,1.750000,396.289375,1.571429,2.666667,0.666667,4.666667
3,4,5YJ00000000000004,Model 3,2019,LFP,2025-12-31,North America,87159,Retail,60,2.0,0.16,High,3.0,0.0,0.0,0.0,1760.020000,3445.09

In [58]:
# High-risk flags

vehicle_features["high_repair_frequency_flag"] = (vehicle_features["repairs_last_90d"] >= 3).astype(int)
vehicle_features["high_warning_flag"] = (vehicle_features["avg_warning_count"] >= 4).astype(int)
vehicle_features["low_battery_health_flag"] = (vehicle_features["avg_battery_health"] < 85).astype(int)
vehicle_features["high_claim_cost_flag"] = (vehicle_features["total_claim_amount"] > vehicle_features["total_claim_amount"].quantile(0.90)).astype(int)
vehicle_features["repeat_repair_flag"] = (vehicle_features["repeat_repair_ratio"] > 0.30).astype(int)

vehicle_features[[
    "vehicle_id",
    "high_repair_frequency_flag",
    "high_warning_flag",
    "low_battery_health_flag",
    "high_claim_cost_flag",
    "repeat_repair_flag"
]].head()

/tmp/ipykernel_174/1798653485.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  vehicle_features["high_repair_frequency_flag"] = (vehicle_features["repairs_last_90d"] >= 3).astype(int)
/tmp/ipykernel_174/1798653485.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  vehicle_features["high_warning_flag"] = (vehicle_features["avg_warning_count"] >= 4).astype(int)
/tmp/ipykernel_174/1798653485.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor

,vehicle_id,high_repair_frequency_flag,high_warning_flag,low_battery_health_flag,high_claim_cost_flag,repeat_repair_flag
0,1,0,0,0,0,0
1,2,0,0,0,0,0
2,3,0,0,0,0,0
3,4,0,0,1,0,1
4,5,0,0,0,0,0


In [59]:
# Create overall risk score

vehicle_features["fleet_risk_score"] = (
    vehicle_features["high_repair_frequency_flag"] +
    vehicle_features["high_warning_flag"] +
    vehicle_features["low_battery_health_flag"] +
    vehicle_features["high_claim_cost_flag"] +
    vehicle_features["repeat_repair_flag"]
)

vehicle_features[["vehicle_id", "fleet_risk_score"]].head()

/tmp/ipykernel_174/165064775.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  vehicle_features["fleet_risk_score"] = (


,vehicle_id,fleet_risk_score
0,1,0
1,2,0
2,3,0
3,4,2
4,5,0


In [61]:
# Shape and null check

print("Final vehicle feature table shape:", vehicle_features.shape)
print()

print("Null values summary:")
print(vehicle_features.isnull().sum().sort_values(ascending=False).head(20))

Final vehicle feature table shape: (50000, 117)

Null values summary:
last_claim_date          6781
last_failure_date        4622
last_service_date         122
last_telemetry_date         1
latest_telemetry_date       1
delivery_date               0
region                      0
mileage                     0
fleet_type                  0
vehicle_age_days            0
vehicle_age_months          0
vehicle_age_years           0
mileage_band                0
total_services              0
repairs_last_30d            0
repairs_last_90d            0
repairs_last_180d           0
avg_repair_cost             0
model                       0
vin                         0
dtype: int64


In [62]:
# Preview Final Feature table

vehicle_features.sample(5)

,vehicle_id,vin,model,model_year,battery_type,delivery_date,region,mileage,fleet_type,vehicle_age_days,vehicle_age_months,vehicle_age_years,mileage_band,total_services,repairs_last_30d,repairs_last_90d,repairs_last_180d,avg_repair_cost,max_repair_cost,total_repair_cost,avg_downtime_days,max_downtime_days,total_downtime_days,repeat_issue_count,last_service_date,days_since_last_service,repeat_repair_ratio,critical_severity_count,high_severity_count,low_severity_count,medium_severity_count,battery_issue_count,brake_issue_count,electrical_issue_count,hvac_issue_count,motor_issue_count,software_issue_count,suspension_issue_count,telemetry_records,avg_battery_health,min_battery_health,avg_motor_temp,max_motor_temp,avg_brake_temp,max_brake_temp,avg_tire_pressure_alerts,total_tire_pressure_alerts,avg_warning_count,max_warning_count,total_warning_count,recent_telemetry_30d,recent_telemetry_90d,last_telemetry_date,days_since_last_telemetry,latest_telemetry_date,latest_battery_health,latest_motor_temp,latest_brake_temp,latest_tire_pressure_alerts,latest_software_version,latest_warning_count,total_claims,claims_last_90d,claims_last_180d,total_claim_amount,avg_claim_amount,max_claim_amount,last_claim_date,days_since_last_claim,approved_claim_count,pending_claim_count,rejected_claim_count,total_failures,failures_last_90d,failures_last_180d,replaced_count,last_failure_date,days_since_last_failure,electrical_fault,mechanical_failure,software_error,thermal_issue,wear,12v_battery,abs_module,ac_compressor,autopilot_computer,bms,battery_pack,brake_pads,brake_rotor,cabin_heater,charging_port,control_arm,cooling_fan,drive_unit,firmware,infotainment_module,inverter,motor_controller,sensor_module,shock_absorber,suspension_link,wiring_harness,claims_per_service,failures_per_service,avg_cost_per_downtime_day,warning_to_failure_ratio,service_per_year,claim_per_year,failure_per_year,high_repair_frequency_flag,high_warning_flag,low_battery_health_flag,high_claim_cost_flag,repeat_repair_flag,fleet_risk_score
40254,40255,5YJ00000000040255,Model Y,2022,NCM,2020-07-18,North America,64999,Retail,2052,68.4,5.62,High,2.0,0.0,0.0,0.0,1589.825,1991.12,3179.65,4.450,6.2,8.9,0.0,2025-01-21,404.0,0.00,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,15.0,91.394000,86.16,70.131333,94.31,58.133333,76.38,0.466667,7.0,1.733333,4.0,26.0,2.0,5.0,2026-02-27,2.0,2026-02-27,90.76,59.21,47.07,0.0,2024.38.1,3.0,1.0,0.0,0.0,729.41,729.410000,729.41,2022-05-04,1397.0,0.0,0.0,1.0,3.0,0.0,0.0,3.0,2025-01-19,406.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.50,1.500000,357.264045,8.666667,0.355872,0.177936,0.533808,0,0,0,0,0,0
31818,31819,5YJ00000000031819,Model Y,2022,NCA,2023-08-27,North America,74370,Retail,917,30.6,2.51,High,5.0,0.0,1.0,2.0,2483.666,5387.28,12418.33,4.100,6.5,20.5,1.0,2025-12-05,86.0,0.20,1.0,1.0,3.0,0.0,0.0,1.0,1.0,1.0,2.0,0.0,0.0,19.0,91.136316,87.40,70.299474,92.72,58.168421,83.93,0.578947,11.0,1.631579,4.0,31.0,1.0,3.0,2026-02-07,22.0,2026-02-07,87.40,66.62,58.10,1.0,2025.2.6,3.0,1.0,0.0,0.0,842.14,842.140000,842.14,2022-10-21,1227.0,1.0,0.0,0.0,4.0,1.0,1.0,3.0,2026-02-25,4.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.20,0.800000,605.772195,7.750000,1.992032,0.398406,1.593625,0,0,0,0,0,0
30286,30287,5YJ00000000030287,Cybertruck,2024,NCA,2019-10-04,Asia Pacific,27405,Retail,2340,78.0,6.41,Moderate,3.0,0.0,0.0,1.0,1419.040,2102.07,4257.12,3.200,5.4,9.6,0.0,2025-09-14,168.0,0.00,0.0,1.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,8.0,96.477500,91.53,73.541250,79.32,59.557500,72.19,0.500000,4.0,2.250000,4.0,18.0,1.0,2.0,2026-02-10,19.0,2026-02-10,98.16,73.02,54.77,2.0,2025.8.1,1.0,3.0,0.0,0.0,4402.16,1467.386667,1795.64,2025-07-31,213.0,2.0,1.0,0.0,2.0,0.0,0.0,1.0,2024-09-28,519.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.00,0.666667,443.450000,9.000000,0.468019,0.468019,0.312012,0,0,0,0,0

In [63]:
# Feature Summary
vehicle_features.describe(include="all")

,vehicle_id,vin,model,model_year,battery_type,delivery_date,region,mileage,fleet_type,vehicle_age_days,vehicle_age_months,vehicle_age_years,mileage_band,total_services,repairs_last_30d,repairs_last_90d,repairs_last_180d,avg_repair_cost,max_repair_cost,total_repair_cost,avg_downtime_days,max_downtime_days,total_downtime_days,repeat_issue_count,last_service_date,days_since_last_service,repeat_repair_ratio,critical_severity_count,high_severity_count,low_severity_count,medium_severity_count,battery_issue_count,brake_issue_count,electrical_issue_count,hvac_issue_count,motor_issue_count,software_issue_count,suspension_issue_count,telemetry_records,avg_battery_health,min_battery_health,avg_motor_temp,max_motor_temp,avg_brake_temp,max_brake_temp,avg_tire_pressure_alerts,total_tire_pressure_alerts,avg_warning_count,max_warning_count,total_warning_count,recent_telemetry_30d,recent_telemetry_90d,last_telemetry_date,days_since_last_telemetry,latest_telemetry_date,latest_battery_health,latest_motor_temp,latest_brake_temp,latest_tire_pressure_alerts,latest_software_version,latest_warning_count,total_claims,claims_last_90d,claims_last_180d,total_claim_amount,avg_claim_amount,max_claim_amount,last_claim_date,days_since_last_claim,approved_claim_count,pending_claim_count,rejected_claim_count,total_failures,failures_last_90d,failures_last_180d,replaced_count,last_failure_date,days_since_last_failure,electrical_fault,mechanical_failure,software_error,thermal_issue,wear,12v_battery,abs_module,ac_compressor,autopilot_computer,bms,battery_pack,brake_pads,brake_rotor,cabin_heater,charging_port,control_arm,cooling_fan,drive_unit,firmware,infotainment_module,inverter,motor_controller,sensor_module,shock_absorber,suspension_link,wiring_harness,claims_per_service,failures_per_service,avg_cost_per_downtime_day,warning_to_failure_ratio,service_per_year,claim_per_year,failure_per_year,high_repair_frequency_flag,high_warning_flag,low_battery_health_flag,high_claim_cost_flag,repeat_repair_flag,fleet_risk_score
count,50000.000000,50000,50000,50000.000000,50000,50000,50000,50000.000000,50000,50000.000000,50000.000000,50000.000000,50000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,49878,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.00000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,49999,50000.000000,49999,50000.00000,50000.000000,50000.000000,50000.000000,50000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,43219,50000.000000,50000.000000,50000.000000,50000.00000,50000.000000,50000.00000,50000.000000,50000.000000,45378,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.00000,50000.000000,50000.000000,50000.000000,50000.00000,50000.00000,50000.00000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
unique,NaN,50000,5,NaN,3,NaN,4,NaN,3,NaN,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,5YJ00000000049984,Model Y,NaN,NCA,NaN,North America,NaN,Retail,NaN,NaN,NaN,Moderate,N

In [64]:
# Save Vehicle-level feature table

vehicle_features.to_csv(f"{feature_path}/vehicle_features.csv", index=False)
vehicle_features.to_parquet(f"{feature_path}/vehicle_features.parquet", index=False)

print("Vehicle-level feature table saved successfully.")

Vehicle-level feature table saved successfully.


In [65]:
# Save intermediate aggregates too

service_agg.to_csv(f"{feature_path}/service_agg.csv", index=False)
telemetry_agg.to_csv(f"{feature_path}/telemetry_agg.csv", index=False)
warranty_agg.to_csv(f"{feature_path}/warranty_agg.csv", index=False)
failure_agg.to_csv(f"{feature_path}/failure_agg.csv", index=False)

print("Intermediate aggregate tables saved successfully.")

Intermediate aggregate tables saved successfully.


## Output
A final `vehicle_features` table that can be used for:
- KPI analysis
- dashboard development
- anomaly detection
- machine learning models